# CI-Triage-Env — GRPO Training Notebook

Colab-runnable end-to-end training pipeline:
1. Install dependencies
2. Pull scenario corpus from HF Hub
3. Start env server
4. SFT warmstart on C3 trajectory dataset
5. GRPO smoke test (100 steps)
6. Full GRPO (3000 steps)
7. Push adapter to HF Hub

**Prerequisites**: `HF_TOKEN`, `OPENAI_API_KEY`, `WANDB_API_KEY` set as Colab secrets.

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q unsloth trl transformers accelerate peft
!pip install -q wandb datasets huggingface_hub openai httpx fastapi uvicorn pydantic jsonschema
!pip install -q -e .  # install ci_triage_env package in editable mode

In [ ]:
# Cell 2: Environment setup
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['WANDB_PROJECT'] = 'ci-triage-env'

import wandb
wandb.login()

In [ ]:
# Cell 3: Pull scenario corpus from HF dataset hub
# Replace YOUR_ORG with your HuggingFace org/username
HF_DATASET_REPO = 'YOUR_ORG/ci-triage-scenarios'
HF_MODEL_REPO  = 'YOUR_ORG/ci-triage-trained-qwen3.5-4b'

from huggingface_hub import snapshot_download
scen_dir = snapshot_download(HF_DATASET_REPO, repo_type='dataset',
                             local_dir='data_artifacts/scenarios')
print(f'Scenarios downloaded to {scen_dir}')

In [ ]:
# Cell 4: Start env server in background
import subprocess, time
server_proc = subprocess.Popen(
    ['python', '-m', 'ci_triage_env.env.server'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)  # give server time to start
print('Env server started, PID:', server_proc.pid)

In [ ]:
# Cell 5: Generate SFT trajectories (skip if already done)
import os
if not os.path.exists('data_artifacts/sft_dataset'):
    from ci_triage_env.training.trajectory_gen import main as traj_main
    traj_main([
        '--count', '600',
        '--model', 'gpt-4o-mini',
        '--budget', '25.0',
        '--output', 'data_artifacts/sft_dataset/',
    ])
else:
    print('SFT dataset already exists, skipping generation.')

In [ ]:
# Cell 6: SFT warmstart
from ci_triage_env.training.sft import run_sft

run_sft(
    dataset_path='data_artifacts/sft_dataset/',
    output_dir='checkpoints/sft/',
    num_epochs=3,
    per_device_batch_size=1,
    gradient_accumulation_steps=4,
)
print('SFT complete. Checkpoint at checkpoints/sft/')

In [ ]:
# Cell 7: GRPO smoke test (100 steps, ~30 min)
from ci_triage_env.training.grpo import run_grpo

run_grpo(
    sft_checkpoint_dir='checkpoints/sft/',
    output_dir='checkpoints/grpo_smoke/',
    total_steps=100,
)
print('Smoke test complete. Check W&B for reward curve.')

In [ ]:
# Cell 8: Full GRPO run (3000 steps, ~30h wall-clock)
# Monitor: https://wandb.ai/<entity>/ci-triage-env
# Hard-stop rules: see plan/branch-c-reward-training/phase-c4.md
run_grpo(
    sft_checkpoint_dir='checkpoints/sft/',
    output_dir='checkpoints/grpo_full/',
    total_steps=3000,
)
print('Full GRPO complete.')

In [ ]:
# Cell 9: Push trained adapter to HF Hub
from huggingface_hub import upload_folder

upload_folder(
    folder_path='checkpoints/grpo_full/',
    repo_id=HF_MODEL_REPO,
    repo_type='model',
    commit_message='CI-Triage-Env GRPO-trained Qwen3.5-4B adapter',
)
print(f'Adapter pushed to https://huggingface.co/{HF_MODEL_REPO}')